In [1]:
from pathlib import Path
import sys

In [2]:

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(root_dir / "src"))
root_dir


PosixPath('/Volumes/256 SSD/deal_hunter')

In [3]:
from deal_hunter.config import settings

In [4]:
settings.rss_feed_url

['https://www.dealnews.com/c142/Electronics/?rss=1',
 'https://www.dealnews.com/c39/Computers/?rss=1',
 'https://www.dealnews.com/f1912/Smart-Home/?rss=1']

### Testing deals.py , pydantic models

In [5]:
from deal_hunter.models import ScrapedDeal, Deal,DealSelection, Opportunity

In [6]:
#making class
sd = ScrapedDeal(
    title = "Test Product" * 20,
    summary = "test SUmmary",
    url = "https://example.com",
    details = "some details" * 100,
    features = " Feature List" * 100
)



In [7]:
#checking methods
sd.truncate()
print(repr(sd))
print(sd.describe())
print(len(sd.title))
print(len(sd.details))



<Test ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest>
Title: Test ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest ProductTest
Details: some detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome detailssome det
Features: Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature List Feature L

In [8]:
# Verifying other models
d = Deal(product_description="A widget", price=29.99, url="https://example.com")
ds = DealSelection(deals=[d])
o = Opportunity(deal=d, estimate=50.0, discount=0.40)
print(o)

deal=Deal(product_description='A widget', price=29.99, url='https://example.com') estimate=50.0 discount=0.4


In [9]:
from deal_hunter.services import Rss_Service

rss = Rss_Service()
deals = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    max_per_feed=3,
)
print(f"Scraped {len(deals)} deals")
for d in deals[:3]:
    print(d.describe())
    print()

Scraped 9 deals
Title: Shokz OpenRun Pro Bluetooth Bone Conduction Sport Headphones for $125 + free shipping
Details: more


That's the second-lowest these popular jogging headphones have been.  
Buy Now at Amazon
Features: sweat resistant
 
10-hour battery life on a single charge
 
dual noise-canceling microphones
 
Model: S810
URL:https://www.dealnews.com/products/Shokz/Shokz-Open-Run-Pro-Bluetooth-Bone-Conduction-Sport-Headphones/409928.html?iref=rss-c142

Title: Unlocked Samsung Galaxy S26 Ultra Android Smartphone: $200 off or up to $720 off w/ trade + free shi
Details: more


Trade in an eligible device, and you can get up to $720 off, dropping starting prices to $580. Alternatively, those without a trade can save $200 off, plus get a $100 credit to use towards the purchase of accessories. 
Shop Now at Samsung
Features: 
URL:https://www.dealnews.com/Unlocked-Samsung-Galaxy-S26-Ultra-Android-Smartphone-200-off-or-up-to-720-off-w-trade-free-shipping/21821967.html?iref=rss-c142

Titl

In [10]:
already_seen = {d.url for d in deals}
print(f"Known URLs: {len(already_seen)}")

deals_round2 = rss.scrape_feeds(
    feed_urls=settings.rss_feed_url,
    known_urls=already_seen,
    max_per_feed=3,
)
print(f"Round 2: {len(deals_round2)} new deals (should be 0 or very few)")

Known URLs: 9
Round 2: 0 new deals (should be 0 or very few)


### testing notification service

In [11]:
from deal_hunter.config import settings
from deal_hunter.services.notifications import PushoverNotifier

notifier = PushoverNotifier(
    user_key=settings.pushover_user,
    token=settings.pushover_token,
    url=settings.pushover_url,
)

print("user set?", bool(settings.pushover_user))
print("token set?", bool(settings.pushover_token))
print("url:", settings.pushover_url)

user set? True
token set? True
url: https://api.pushover.net/1/messages.json


In [12]:
ok = notifier.send("Phase 4 test from scanning.ipynb")
print("send() returned:", ok)

send() returned: True


In [13]:
from deal_hunter.config import settings

print("user set?", bool(settings.pushover_user), "len:", len(settings.pushover_user))
print("token set?", bool(settings.pushover_token), "len:", len(settings.pushover_token))
print("url:", settings.pushover_url)


user set? True len: 30
token set? True len: 30
url: https://api.pushover.net/1/messages.json


In [14]:
import requests
from deal_hunter.config import settings

payload = {
    "user": settings.pushover_user,
    "token": settings.pushover_token,
    "message": "debug test from notebook",
    "sound": "cashregister",
}

resp = requests.post(settings.pushover_url, data=payload, timeout=10)
print("HTTP status:", resp.status_code)
print("Response text:", resp.text)

HTTP status: 200
Response text: {"status":1,"request":"3f3f1595-0e96-47a6-9f10-6a1a0b8c16fd"}
